# Raw CCTV behavior pilot — civet (Camera 3)

End-to-end pipeline on raw zoo CCTV: **detect the civet → classify its behavior → add motion → write an observation sheet.**

```
MegaDetector V6  →  ResNet18 CNN             →  motion gate                 →  observation sheet
(civet box)         (Eating/Locomotion/Resting)  (fixes eating read as rest)    (time, status, behavior)
```

## How to run (top to bottom)
1. **§1 — Setup.** Install, mount Drive, pull the latest code. *First: Runtime → Change runtime type → **GPU**.*
2. **§2 — Pick a clip.** Uncomment ONE clip line; set the model path.
3. **§3 — Model.** Trains `model.pt` once (skips if it already exists).
4. **§4 — Detect.** Sample 1 frame/second and run the detector.
5. **§5 — Classify.** Run the CNN on the crops; save the raw-CNN preview.
6. **§6 — Motion gate + sheet.** Apply the motion gate; show the final observation sheet + gated preview.

## What to expect (and say plainly in the talk)
- **Resting, daytime** → confident and correct.
- **Eating** → the single-frame CNN often calls it *Resting* (a still, hunched eating civet looks like a resting one). **§6's motion gate corrects this** using frame-to-frame box motion.
- **Infrared / unfamiliar camera views** → weak; see the limitations in `WRITEUP.md`.

> Keep the source video and every output in your private Drive. **Never commit footage.**


## §1 — Setup: mount Drive and pull the code

**Set the runtime to GPU first** (Runtime → Change runtime type → GPU), then run the two cells below. They mount your Drive, fetch the latest project scripts from GitHub, and set the private data paths. The raw video is never cloned from GitHub.


In [1]:
# Install the wildlife detector package
%pip -q install PytorchWildlife opencv-python-headless


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess, sys, json, math, shutil, os
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw
from IPython.display import display, Video

DRIVE_MYDRIVE = Path("/content/drive/MyDrive")
SHARED_PROJECT_NAME = "CMZprojectphase2"
PROJECT_ROOT = DRIVE_MYDRIVE / SHARED_PROJECT_NAME
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Shared project folder not found: {PROJECT_ROOT}. "
        "In Google Drive, add a shortcut to the shared CMZprojectphase2 folder under My Drive, "
        "or update SHARED_PROJECT_NAME to the shortcut's name."
    )
print("Shared CMZ project:", PROJECT_ROOT)

drive_root = PROJECT_ROOT
repo = Path("/content/cmz-project")
branch = "feat/civet-zoo-rules-cvat"
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", str(repo), "reset", "--hard", f"origin/{branch}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", branch, "https://github.com/Siwali/cmz-project.git", str(repo)], check=True)

PIPE = repo / "pipeline"
sys.path.insert(0, str(PIPE))
from behavior_data import tight_crop, CLASSES

# Daytime A03 source video (recording started 14:03:27). OUT is set per-clip in section 2.
RAW = drive_root / "data" / "raw"
VIDEO_PATH = RAW / "cam3" / "A03_20260405140327.mp4"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
print("Video:", VIDEO_PATH, "exists:", VIDEO_PATH.exists())
if not VIDEO_PATH.exists():
    raise FileNotFoundError("Update VIDEO_PATH to the raw MP4 in your private Drive.")


Mounted at /content/drive
Shared CMZ project: /content/drive/MyDrive/CMZprojectphase2
GPU: not available
Video: /content/drive/MyDrive/CMZprojectphase2/data/raw/cam3/A03_20260405140327.mp4 exists: True


## §2 — Pick a clip and confidence settings

Times are elapsed from the start of the source video; `CCTV_START` maps elapsed second 0 to the real clock (14:03:27 for this file), so the overlay shows true clock time. **Uncomment exactly one clip.** The detector keeps boxes above `LOW_REVIEW_CONF`; the CNN classifies only above `CLASSIFY_CONF`. If too many seconds come back `Unknown`, lower `CLASSIFY_CONF`.


In [3]:
# ===== §2  Pick ONE clip: uncomment its line, leave the others commented =====
# Source A03_20260405140327.mp4 started recording at 14:03:27, so elapsed second 0 = that clock.
CCTV_START = "2026-04-05 14:03:27"   # real clock at elapsed second 0 of the video

# Clip A - daytime EATING (walks up -> reaches food -> chews). ~145 s. Shows the motion gate working.
CLIP_NAME, CLIP_START_S, CLIP_END_S = "day_eating", 6330, 6475      # 1:45:30 - 1:47:53

# Clip B - daytime RESTING (true still rest, head on body). 120 s. The clean "it works" demo.
# CLIP_NAME, CLIP_START_S, CLIP_END_S = "day_resting", 7640, 7760   # 2:07:20 - 2:09:20

# Clip C - full arc Eat -> Rest -> Eat for a status timeline. ~2080 s; GPU only (slow). Run for the CSV.
# CLIP_NAME, CLIP_START_S, CLIP_END_S = "day_arc", 6330, 8410       # 1:45:30 - 2:20:10

# Clip D - daytime EATING #2 (second feeding bout, calmer chewing). ~120 s. Gate partly recovers it.
# CLIP_NAME, CLIP_START_S, CLIP_END_S = "day_eating2", 8250, 8370    # 2:17:30 - 2:19:30

# Clip E - daytime RESTING #2 (deeper into the sleep, head down, still). ~120 s. Clean "it works".
# CLIP_NAME, CLIP_START_S, CLIP_END_S = "day_resting2", 7900, 8020   # 2:11:40 - 2:13:40

# Clip F - daytime LOCOMOTION (circling the tire bed). ~47 s. CNN alone misses it; the gate recovers ~60%.
# CLIP_NAME, CLIP_START_S, CLIP_END_S = "day_locomotion", 7120, 7167 # 1:58:40 - 1:59:27

SAMPLE_FPS = 1
LOW_REVIEW_CONF = 0.15          # keep detector boxes above this for review
CLASSIFY_CONF = 0.80            # classify only above this; lower to ~0.5 if too many Unknown seconds

# Reviewed fixed-split checkpoint (trained with review_edits.csv). §3 trains it if missing.
MODEL_DIR = drive_root / "data" / "raw_pilot_model_reviewed"
MODEL_PATH = MODEL_DIR / "model.pt"

OUT = drive_root / "data" / f"raw_pilot_{CLIP_NAME}"
OUT.mkdir(parents=True, exist_ok=True)
FRAME_DIR = Path("/content/raw_pilot_frames")
FRAME_DIR.mkdir(parents=True, exist_ok=True)

print(f"Clip '{CLIP_NAME}': {CLIP_START_S}s to {CLIP_END_S}s; expected about {CLIP_END_S-CLIP_START_S} frames")
print("Output:", OUT)
print("Checkpoint exists:", MODEL_PATH.exists(), MODEL_PATH)


Clip 'day_eating': 6330s to 6475s; expected about 145 frames
Output: /content/drive/MyDrive/CMZprojectphase2/data/raw_pilot_day_eating
Checkpoint exists: True /content/drive/MyDrive/CMZprojectphase2/data/raw_pilot_model_reviewed/model.pt


## §3 — Model: train a checkpoint (only if missing)

Cross-validation reports accuracy but saves no single model. This cell trains the fixed train/val split and saves `model.pt` to Drive. It **skips automatically if `model.pt` already exists**, so you normally run it once. It needs the processed ZIP and, if present, `redetect.csv` and `review_edits.csv` in `data/` (the latter is the reviewed-label cleanup).


In [ ]:
# Optional: run only when MODEL_PATH does not already exist.
if not MODEL_PATH.exists():
    import zipfile
    ZIP_PATH = drive_root / "data" / "cmz_processed_dataset.zip"
    BASE = Path("/content/CMZ")
    manifest = BASE / "metadata" / "manifest.csv"
    if not manifest.exists():
        if not ZIP_PATH.exists():
            raise FileNotFoundError(f"Processed dataset ZIP not found: {ZIP_PATH}")
        BASE.mkdir(parents=True, exist_ok=True)
        print("Extracting processed dataset to local Colab storage; this happens once per runtime...")
        with zipfile.ZipFile(ZIP_PATH) as zf:
            zf.extractall(BASE)
    else:
        print("Reusing extracted dataset:", BASE)
    if not manifest.exists():
        raise FileNotFoundError("ZIP did not extract to /content/CMZ/metadata/manifest.csv; inspect its internal folder layout.")
    drive_redetect = drive_root / "data" / "redetect.csv"
    if drive_redetect.exists():
        shutil.copy2(drive_redetect, BASE / "metadata" / "redetect.csv")
    edits = drive_root / "data" / "review_edits.csv"
    cmd = [sys.executable, "-u", str(PIPE / "train_behavior.py"),
           "--base", str(BASE), "--out", str(MODEL_DIR),
           "--epochs", "4", "--freeze-backbone", "--redetect-min-conf", "0.35"]
    if edits.exists():
        cmd += ["--edits", str(edits)]
    print("Training the final fixed-split model. Keep this cell running until it prints that outputs were saved.")
    subprocess.run(cmd, check=True)
else:
    print("Using existing checkpoint:", MODEL_PATH)

if not MODEL_PATH.exists():
    raise FileNotFoundError("Training finished without creating model.pt. Check the training output above.")
print("Checkpoint ready:", MODEL_PATH)


## §4 — Detect: sample 1 fps and run MegaDetector

Extracts one frame per second from the chosen interval to Colab's local disk, then runs the civet detector in batches. The raw video stays in Drive. The low-confidence CSV lists possible detections for review; it is not automatically treated as the civet.


In [ ]:
from PytorchWildlife.models import detection as pw_detection
import cv2
from datetime import datetime, timedelta

# Extract sampled frames from the requested source interval.
shutil.rmtree(FRAME_DIR, ignore_errors=True)
FRAME_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
    "-ss", str(CLIP_START_S), "-i", str(VIDEO_PATH),
    "-t", str(CLIP_END_S - CLIP_START_S),
    "-vf", f"fps={SAMPLE_FPS}", "-q:v", "3",
    str(FRAME_DIR / "frame_%06d.jpg")
], check=True)
frame_paths = sorted(FRAME_DIR.glob("*.jpg"))
if not frame_paths:
    raise RuntimeError("No sampled frames extracted; check the video and interval.")
print("Extracted frames:", len(frame_paths))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
md = pw_detection.MegaDetectorV6Apache(device=DEVICE, pretrained=True, version="MDV6-apa-rtdetr-e")
batch_size = 8 if DEVICE == "cuda" else 2
results = md.batch_image_detection(str(FRAME_DIR), batch_size=batch_size)
if len(results) != len(frame_paths):
    raise RuntimeError(f"Detector returned {len(results)} results for {len(frame_paths)} frames")
print("Detector done on", DEVICE)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


Extracted frames: 145


## §5 — Classify the civet crops (raw CNN)

Runs the ResNet18 classifier on each high-confidence crop and writes `raw_frame_predictions.csv` plus a raw-CNN preview. Crop size, padding, normalization, and class order match training.

**Heads-up:** on *eating* clips the single-frame CNN tends to output **Resting** (a still, hunched eating civet looks like a resting one), often at low confidence. Don't judge the result here — **§6 adds motion and corrects it.** If too many seconds come back `Unknown`, lower `CLASSIFY_CONF` in §2.


In [ ]:
import torch.nn as nn
from torchvision import models, transforms as T

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing classifier checkpoint: {MODEL_PATH}")
classifier = models.resnet18(weights=None)
classifier.fc = nn.Linear(classifier.fc.in_features, len(CLASSES))
state = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
classifier.load_state_dict(state)
classifier.to(DEVICE).eval()
tf = T.Compose([T.ToTensor(), T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])])

per_frame, low_candidates = [], []
for frame_idx, (frame_path, result) in enumerate(zip(frame_paths, results)):
    frame = cv2.imread(str(frame_path))
    if frame is None:
        continue
    det = result.get("detections")
    boxes = np.asarray(det.xyxy) if det is not None else np.empty((0,4))
    scores = np.asarray(getattr(det, "confidence", [])).reshape(-1) if det is not None else np.array([])
    cats = np.asarray(getattr(det, "class_id", [])).reshape(-1) if det is not None else np.array([])
    animal = [(b, float(s)) for b,s,c in zip(boxes, scores, cats) if int(c)==0 and float(s)>=LOW_REVIEW_CONF]
    animal.sort(key=lambda x:x[1], reverse=True)
    elapsed = CLIP_START_S + frame_idx / SAMPLE_FPS
    clock = (datetime.fromisoformat(CCTV_START) + timedelta(seconds=elapsed)).strftime("%H:%M:%S")
    row = {"elapsed_s": elapsed, "cctv_time": clock, "md_conf": animal[0][1] if animal else 0.0,
           "behavior": "Unknown", "behavior_conf": 0.0, "p_Eating": 0.0, "p_Locomotion": 0.0, "p_Resting": 0.0,
           "x1": np.nan, "y1": np.nan, "x2": np.nan, "y2": np.nan}
    if animal:
        box, conf = animal[0]
        x1,y1,x2,y2 = map(float, box)
        row.update(x1=x1,y1=y1,x2=x2,y2=y2)
        low_candidates.append({"elapsed_s":elapsed,"cctv_time":clock,"md_conf":conf,"x1":x1,"y1":y1,"x2":x2,"y2":y2})
        if conf >= CLASSIFY_CONF:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            crop = tight_crop(Image.fromarray(rgb), (x1,y1,x2,y2), margin=0.1)
            with torch.no_grad():
                probs = torch.softmax(classifier(tf(crop).unsqueeze(0).to(DEVICE)), dim=1)[0].cpu().numpy()
            k = int(probs.argmax())
            row.update(behavior=CLASSES[k], behavior_conf=float(probs[k]))
            for cls,p in zip(CLASSES, probs): row[f"p_{cls}"] = float(p)
    per_frame.append(row)

pred = pd.DataFrame(per_frame)
pred.to_csv(OUT / "raw_frame_predictions.csv", index=False)
pd.DataFrame(low_candidates).to_csv(OUT / "low_confidence_detection_candidates.csv", index=False)
print(pred["behavior"].value_counts(dropna=False).to_string())
print("Saved frame predictions and review candidates to", OUT)


In [ ]:
# Render the sampled frames with the box/behavior overlay, then re-encode to H.264 so it plays inline.
import shutil, subprocess

first = cv2.imread(str(frame_paths[0]))
if first is None:
    raise FileNotFoundError(f"Could not read first frame: {frame_paths[0]}")
h, w = first.shape[:2]

tmp_raw  = Path("/content/raw_behavior_preview.mp4")        # write to local disk first
tmp_h264 = Path("/content/raw_behavior_preview_h264.mp4")

writer = cv2.VideoWriter(str(tmp_raw), cv2.VideoWriter_fourcc(*"mp4v"), SAMPLE_FPS, (w, h))
if not writer.isOpened():
    raise RuntimeError("VideoWriter failed to open")

written = 0
for frame_path, row in zip(frame_paths, per_frame):
    frame = cv2.imread(str(frame_path))
    if frame is None:
        continue
    if frame.shape[:2] != (h, w):
        frame = cv2.resize(frame, (w, h))
    if np.isfinite(row["x1"]):
        color = (0,220,0) if row["md_conf"] >= CLASSIFY_CONF else (0,180,255)
        cv2.rectangle(frame, (int(row["x1"]),int(row["y1"])), (int(row["x2"]),int(row["y2"])), color, 3)
    text = f"{row['cctv_time']} | {row['behavior']} {row['behavior_conf']:.2f} | MD {row['md_conf']:.2f}"
    cv2.rectangle(frame, (0,0), (min(w,1100),38), (0,0,0), -1)
    cv2.putText(frame, text, (10,27), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255,255,255), 2)
    writer.write(frame)
    written += 1
writer.release()
print(f"Wrote {written}/{len(frame_paths)} frames, file size {tmp_raw.stat().st_size:,} bytes")
if written == 0:
    raise RuntimeError("No frames were written")

# Re-encode to H.264 so the browser can play it (mp4v often renders a blank preview in Colab).
r = subprocess.run([
    "ffmpeg", "-y", "-i", str(tmp_raw),
    "-an", "-c:v", "libx264", "-preset", "fast",
    "-pix_fmt", "yuv420p", "-movflags", "+faststart",
    str(tmp_h264),
], capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-2000:])
    raise RuntimeError(f"ffmpeg failed ({r.returncode})")

# Copy both versions to the shared output folder; the H.264 file is the one to show/present.
preview_path = OUT / "raw_behavior_preview.mp4"
h264_path    = OUT / "raw_behavior_preview_h264.mp4"
shutil.copy(tmp_raw, preview_path)
shutil.copy(tmp_h264, h264_path)
print("Saved:", h264_path)

display(Video(str(tmp_h264), embed=True, width=960))

## §6 — Motion gate + observation sheet

A single-frame CNN confuses a stationary *eating* civet with a *resting* one. A resting civet is still, though, while an eating one moves (head, paws, chewing), so we measure the frame-to-frame motion inside the civet box and re-call any "Resting" second whose motion exceeds a threshold as Active. This step applies that gate, smooths the per-second labels, collapses them into behavior bouts (`time / status / behavior`), and shows a gated preview — this is the **final output**.

The threshold (`--motion-thresh 4.5`) is hand-calibrated on daytime 1-fps clips of this camera: true rest ≈ 3.5, eating ≈ 5–8, grooming overlaps eating. 4.5 catches most eating and treats grooming as active. It is **not** the dataset's 0.88 CNN+motion stacker; recalibrate for other sampling rates or infrared.


In [ ]:
# Apply the motion gate, build the gated observation sheet, and show a gated preview.
subprocess.run([sys.executable, str(PIPE / "apply_motion_gate.py"),
                "--run", str(OUT), "--frames-dir", str(FRAME_DIR),
                "--motion-thresh", "4.5"], check=True)

sheet = pd.read_csv(OUT / "observation_sheet_gated.csv")
print("Motion-gated observation sheet:")
display(sheet)

# Re-encode the gated preview to H.264 so it plays inline in Colab (mp4v shows blank).
gated_mp4v = OUT / "raw_behavior_preview_gated.mp4"
gated_h264 = Path("/content/raw_behavior_preview_gated_h264.mp4")
r = subprocess.run(["ffmpeg", "-y", "-i", str(gated_mp4v), "-an", "-c:v", "libx264",
                    "-preset", "fast", "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(gated_h264)],
                   capture_output=True, text=True)
if r.returncode == 0:
    shutil.copy(gated_h264, OUT / "raw_behavior_preview_gated_h264.mp4")
    display(Video(str(gated_h264), embed=True, width=960))
else:
    print(r.stderr[-1500:])


## What you get, and how to read it

The final output is **§6's** `observation_sheet_gated.csv` — bouts of `time | status | behavior` — plus the gated preview. The per-second `raw_frame_predictions_gated.csv` keeps the raw CNN label, the motion value, and the gated label side by side for inspection.

**Two-clip demo (recommended):**
- **Eating clip (§2 Clip A):** the raw CNN reads mostly *Resting*; the motion gate flips it to a continuous **Active / Eating** bout → shows the gate working.
- **Resting clip (§2 Clip B):** stays **Inactive / Resting** throughout → the clean baseline.

**Be honest about the limits:** calm eating and grooming have overlapping motion, so the gate can't split every case; infrared and unfamiliar camera views are weak; `Unknown` (no detection) is never counted as Resting. Full numbers and future work are in `WRITEUP.md`.

The motion threshold (4.5) is hand-calibrated for **daytime, 1 fps** — recalibrate for coarser sampling or infrared, where sensor noise raises the motion floor.
